# Search Pipeline: Multi-vector + BM25 + CLIP + Query Expansion

In [16]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import json, numpy as np, faiss, time, re, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# --- Данные ---
with open("data/processed/index_metadata.jsonl") as f:
    metadata = [json.loads(l) for l in f]
print(f"Мемов: {len(metadata)}")

# --- FAISS ---
idx_caption = faiss.read_index("data/processed/faiss_caption.index")
idx_ocr = faiss.read_index("data/processed/faiss_ocr.index")
idx_keywords = faiss.read_index("data/processed/faiss_keywords.index")
idx_image = faiss.read_index("data/processed/faiss_image.index")
print("FAISS загружен")

# --- BM25 ---
def tokenize(t):
    return re.findall(r"[a-zа-яё0-9]+", t.lower())

corpus = []
for m in metadata:
    parts = [m.get("caption",""), m.get("ocr_text",""), m.get("main_idea","")]
    objs = m.get("objects", [])
    parts.append(" ".join(str(o) for o in objs) if objs else "")
    parts.append(m.get("tone",""))
    corpus.append(tokenize(" ".join(parts)))
bm25 = BM25Okapi(corpus)
print("BM25 готов")

# --- bge-m3 (CPU чтобы не крашилось) ---
print("Загрузка bge-m3 (CPU)...")
text_model = SentenceTransformer("BAAI/bge-m3", device="cpu")
print("bge-m3 готова")

# --- CLIP ---
from transformers import CLIPModel, CLIPProcessor
print("Загрузка CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP готов")

# --- Переводчик ---
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source="ru", target="en")
print("Переводчик готов")

print()
print("Всё загружено!")


Мемов: 9770
FAISS загружен
BM25 готов
Загрузка bge-m3 (CPU)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

bge-m3 готова
Загрузка CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP готов
Переводчик готов

Всё загружено!


## Функции поиска

In [17]:
def rrf_fusion(ranked_lists, k=60):
    scores = {}
    for rl in ranked_lists:
        for rank, did in enumerate(rl):
            scores[did] = scores.get(did, 0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def clip_text_search(query, top_k=50):
    inputs = clip_proc(text=[query], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = clip_model.get_text_features(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        if not isinstance(out, torch.Tensor):
            out = out.pooler_output
        out = out / out.norm(dim=-1, keepdim=True)
    q = out.numpy().astype(np.float32)
    _, indices = idx_image.search(q, top_k)
    return indices[0].tolist()


def translate_ru_en(text):
    latin = sum(1 for c in text if c.isascii() and c.isalpha())
    total = sum(1 for c in text if c.isalpha())
    if total > 0 and latin / total > 0.5:
        return text
    try:
        return translator.translate(text)
    except:
        return text


def search_v2(query, top_k=10, top_n=50, use_clip=True, use_translate=True):
    t0 = time.time()
    ranked_lists = []
    
    queries_to_search = [query]
    if use_translate:
        tr = translate_ru_en(query)
        if tr != query:
            queries_to_search.append(tr)
    
    for qt in queries_to_search:
        qe = text_model.encode([qt], normalize_embeddings=True).astype(np.float32)
        _, ci = idx_caption.search(qe, top_n); ranked_lists.append(ci[0].tolist())
        _, oi = idx_ocr.search(qe, top_n); ranked_lists.append(oi[0].tolist())
        _, ki = idx_keywords.search(qe, top_n); ranked_lists.append(ki[0].tolist())
        ranked_lists.append(np.argsort(-bm25.get_scores(tokenize(qt)))[:top_n].tolist())
    
    if use_clip:
        ranked_lists.append(clip_text_search(query, top_n))
    
    fused = rrf_fusion(ranked_lists)
    elapsed = time.time() - t0
    
    results = []
    for doc_idx, score in fused[:top_k]:
        m = metadata[doc_idx]
        results.append({
            "rank": len(results) + 1,
            "score": round(score, 4),
            "filename": m["filename"],
            "caption": m.get("caption", "")[:100],
            "ocr": m.get("ocr_text", "")[:60],
        })
    return results, elapsed

print("Функции готовы")


Функции готовы


## Демо поиска

In [18]:
demos = [
    "миленький хомяк пьёт кофе",
    "change my mind",
    "грустная лягушка",
    "npc angry face",
    "когда еда в лучшей форме чем ты",
]

for q in demos:
    r, t = search_v2(q)
    print(f"\"{q}\" ({t*1000:.0f}ms)")
    for x in r[:3]:
        print(f"  {x['rank']}. [{x['score']:.4f}] {x['filename']} — {x['caption'][:60]}")
    print()


"миленький хомяк пьёт кофе" (623ms)
  1. [0.1120] 7877562d187d.png — A cute hamster drinking coffee, enjoying a cozy moment.
  2. [0.0809] 7877562d187d.png — A cute hamster is drinking coffee, but the coffee is not rea
  3. [0.0771] ac5f32369c85.png — A cute animal holding a coffee cup, seemingly enjoying a bev

"change my mind" (102ms)
  1. [0.0694] 4311a8519536.jpg — A man sitting at a table with a sign that reads '共和党 = 民主党 C
  2. [0.0690] 67d58e0a8bdf.jpg — A person is sitting at a table with a sign that says 'CHANGE
  3. [0.0672] a177b98a009e.jpg — A person sitting at a table with a sign that says 'Change My

"грустная лягушка" (334ms)
  1. [0.0777] aaed1f5a2f9f.jpg — A green frog with a sad expression.
  2. [0.0732] 05594b6f229b.png — A pixelated frog with a sad expression.
  3. [0.0728] 657d17ab3a0d.jpg — A sad frog with a blue shirt, looking down.

"npc angry face" (97ms)
  1. [0.0477] 78ed45da8d98.jpg — A simple, angry face with a neutral tone.
  2. [0.0462] d08bf890a782.png —

## Оценка на 100 запросах

In [19]:
with open("eval/language_experiment_queries.json") as f:
    queries = json.load(f)
with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_records = [json.loads(l) for l in f]

old_to_new = {}
ni = 0
for oi, r in enumerate(all_records):
    if not r.get("is_nsfw", False):
        old_to_new[oi] = ni; ni += 1

valid_queries = []
for q in queries:
    nn = old_to_new.get(q["index"])
    if nn is not None:
        qc = dict(q); qc["index"] = nn; valid_queries.append(qc)

# Pre-encode
en_texts = [q.get("query_en","").strip() or q["caption"][:100] for q in valid_queries]
ru_texts = [q.get("query_ru","").strip() or q["caption"][:100] for q in valid_queries]

print("Encode EN...")
en_embs = text_model.encode(en_texts, normalize_embeddings=True, batch_size=16).astype(np.float32)
print("Encode RU...")
ru_embs = text_model.encode(ru_texts, normalize_embeddings=True, batch_size=16).astype(np.float32)

# Translate RU -> EN и encode
print("Translate RU -> EN...")
ru_translated = [translate_ru_en(t) for t in ru_texts]
print("Encode translated...")
ru_tr_embs = text_model.encode(ru_translated, normalize_embeddings=True, batch_size=16).astype(np.float32)

# CLIP encode
print("CLIP encode EN...")
clip_en = []
for t in en_texts:
    inputs = clip_proc(text=[t], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = clip_model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        if not isinstance(out, torch.Tensor):
            out = out.pooler_output
        out = out / out.norm(dim=-1, keepdim=True)
    clip_en.append(out.numpy().flatten())
clip_en_embs = np.array(clip_en, dtype=np.float32)

print("CLIP encode RU...")
clip_ru = []
for t in ru_texts:
    inputs = clip_proc(text=[t], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = clip_model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        if not isinstance(out, torch.Tensor):
            out = out.pooler_output
        out = out / out.norm(dim=-1, keepdim=True)
    clip_ru.append(out.numpy().flatten())
clip_ru_embs = np.array(clip_ru, dtype=np.float32)

print(f"Готово: {len(valid_queries)} запросов")


Encode EN...
Encode RU...
Translate RU -> EN...
Encode translated...
CLIP encode EN...
CLIP encode RU...
Готово: 100 запросов


In [20]:
def evaluate(embs, lang_key, use_bm25=False, use_multi=False, use_clip=False,
             clip_embs=None, use_translate=False, tr_embs=None):
    h1 = h5 = h10 = 0; mrr = 0
    for i, q in enumerate(valid_queries):
        rl = []
        qe = embs[i:i+1]
        _, ci = idx_caption.search(qe, 50); rl.append(ci[0].tolist())
        
        if use_multi:
            _, oi = idx_ocr.search(qe, 50); rl.append(oi[0].tolist())
            _, ki = idx_keywords.search(qe, 50); rl.append(ki[0].tolist())
        
        if use_bm25:
            qt = q.get(lang_key, "").strip() or q["caption"][:100]
            rl.append(np.argsort(-bm25.get_scores(tokenize(qt)))[:50].tolist())
        
        if use_clip and clip_embs is not None:
            ce = clip_embs[i:i+1]
            _, img_ids = idx_image.search(ce, 50)
            rl.append(img_ids[0].tolist())
        
        if use_translate and tr_embs is not None:
            te = tr_embs[i:i+1]
            _, tci = idx_caption.search(te, 50); rl.append(tci[0].tolist())
            if use_multi:
                _, toi = idx_ocr.search(te, 50); rl.append(toi[0].tolist())
            if use_bm25:
                qt_tr = ru_translated[i] if lang_key == "query_ru" else ""
                if qt_tr:
                    rl.append(np.argsort(-bm25.get_scores(tokenize(qt_tr)))[:50].tolist())
        
        fused = rrf_fusion(rl)
        ranking = [d for d, _ in fused]
        t = q["index"]
        if t in ranking:
            p = ranking.index(t) + 1
            if p <= 1: h1 += 1
            if p <= 5: h5 += 1
            if p <= 10: h10 += 1
            if p <= 10: mrr += 1.0 / p
    n = len(valid_queries)
    return h1/n, h5/n, h10/n, mrr/n

print("=" * 75)
print(f"{'Конфигурация':<45} {'Hit@1':>6} {'Hit@5':>6} {'Hit@10':>7} {'MRR':>8}")
print("-" * 75)

configs = [
    ("Caption only (baseline)", en_embs, "query_en", {}),
    ("Multi-vector", en_embs, "query_en", {"use_multi": True}),
    ("BM25 + caption", en_embs, "query_en", {"use_bm25": True}),
    ("Multi+BM25 (EN)", en_embs, "query_en", {"use_bm25": True, "use_multi": True}),
    ("Multi+BM25+CLIP (EN)", en_embs, "query_en", {"use_bm25": True, "use_multi": True, "use_clip": True, "clip_embs": clip_en_embs}),
    ("Multi+BM25 (RU)", ru_embs, "query_ru", {"use_bm25": True, "use_multi": True}),
    ("Multi+BM25+CLIP (RU)", ru_embs, "query_ru", {"use_bm25": True, "use_multi": True, "use_clip": True, "clip_embs": clip_ru_embs}),
    ("Multi+BM25+CLIP+Translate (RU)", ru_embs, "query_ru", {"use_bm25": True, "use_multi": True, "use_clip": True, "clip_embs": clip_ru_embs, "use_translate": True, "tr_embs": ru_tr_embs}),
]

for name, embs, lang, kwargs in configs:
    a, b, c, d = evaluate(embs, lang, **kwargs)
    print(f"{name:<45} {a:>6.0%} {b:>6.0%} {c:>7.0%} {d:>8.4f}")

print()
print("Baseline (MiniLM single caption): EN Hit@5=32%, RU Hit@5=2%")


Конфигурация                                   Hit@1  Hit@5  Hit@10      MRR
---------------------------------------------------------------------------
Caption only (baseline)                          23%    39%     44%   0.2968
Multi-vector                                     17%    41%     49%   0.2716
BM25 + caption                                   28%    47%     53%   0.3584
Multi+BM25 (EN)                                  24%    47%     51%   0.3376
Multi+BM25+CLIP (EN)                             27%    54%     59%   0.3820
Multi+BM25 (RU)                                  15%    37%     45%   0.2382
Multi+BM25+CLIP (RU)                             13%    37%     44%   0.2242
Multi+BM25+CLIP+Translate (RU)                   17%    44%     51%   0.2871

Baseline (MiniLM single caption): EN Hit@5=32%, RU Hit@5=2%
